In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import EarlyStopping,ReduceLROnPlateau
from matplotlib import pyplot as plt

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"keshrivishal21","key":"4c8bb86460f4dd386e694fd83bc26447"}'}

In [ ]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [00:41<00:00, 103MB/s]



In [ ]:
!unzip -q state-farm-distracted-driver-detection.zip -d data

In [ ]:
df = pd.read_csv("/content/data/driver_imgs_list.csv")
df.head()

,subject,classname,img
0,p002,c0,img_44733.jpg
1,p002,c0,img_72999.jpg
2,p002,c0,img_25094.jpg
3,p002,c0,img_69092.jpg
4,p002,c0,img_92629.jpg


In [ ]:
drivers = df['subject'].unique()
print("Total drivers:", len(drivers))

Total drivers: 26


In [ ]:
train_drivers, temp_drivers = train_test_split(
    drivers,
    test_size=0.4,
    random_state=42
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=42
)

print("Train drivers:", len(train_drivers))
print("Val drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

Train drivers: 15
Val drivers: 5
Test drivers: 6


In [ ]:
train_df = df[df['subject'].isin(train_drivers)]
val_df = df[df['subject'].isin(val_drivers)]
test_df = df[df['subject'].isin(test_drivers)]

print(train_df['subject'].nunique())
print(val_df['subject'].nunique())
print(test_df['subject'].nunique())

15
5
6


In [ ]:
base_path = "/content/data/imgs/train"

def add_image_path(df):
    return df.apply(
        lambda row: os.path.join(base_path, row['classname'], row['img']),
        axis=1
    )

train_paths = add_image_path(train_df)
val_paths = add_image_path(val_df)
test_paths = add_image_path(test_df)

train_labels = train_df['classname']
val_labels = val_df['classname']
test_labels = test_df['classname']

In [ ]:
le = LabelEncoder()
train_labels = le.fit_transform(train_labels)
val_labels = le.transform(val_labels)
test_labels = le.transform(test_labels)

In [ ]:
IMG_SIZE = (224,224)
BATCH_SIZE = 16
tf.random.set_seed(42)

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3, try_recover_truncated=True)

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    # 🔥 Updated ROI (for V3)
    top = tf.cast(0.25 * tf.cast(h, tf.float32), tf.int32)
    bottom = tf.cast(0.95 * tf.cast(h, tf.float32), tf.int32)
    left = tf.cast(0.15 * tf.cast(w, tf.float32), tf.int32)
    right = tf.cast(0.85 * tf.cast(w, tf.float32), tf.int32)

    image = image[top:bottom, left:right]

    image = tf.image.resize(image, IMG_SIZE, method='bilinear', antialias=True)
    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(paths, labels, shuffle=True):
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)
    labels = tf.one_hot(labels, depth=10)

    paths = paths.values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=1500, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    return ds


train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

In [ ]:
for path in train_paths[:5]:
    print(path, os.path.exists(path))

/content/data/imgs/train/c0/img_48693.jpg True
/content/data/imgs/train/c0/img_44903.jpg True
/content/data/imgs/train/c0/img_58514.jpg True
/content/data/imgs/train/c0/img_62307.jpg True
/content/data/imgs/train/c0/img_83984.jpg True


In [ ]:
set(train_drivers) & set(val_drivers)
set(train_drivers) & set(test_drivers)
set(val_drivers) & set(test_drivers)

set()

In [ ]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

In [ ]:
print("Train images:", len(train_paths))
print("Val images:", len(val_paths))
print("Test images:", len(test_paths))

In [ ]:
def mobilenetv3_small_model(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    x = base_model(x, training=False)

    # 🔥 Feature extraction
    gap = layers.GlobalAveragePooling2D()(x)
    gmp = layers.GlobalMaxPooling2D()(x)
    x = layers.Concatenate()([gap, gmp])

    # 🔥 Lightweight head
    x = layers.Dense(128,
                     activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model, base_model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
])

In [ ]:
model, base_model = mobilenetv3_small_model()
model.summary()

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

In [ ]:
base_model.trainable = True

# Freeze most layers, train last 40
for layer in base_model.layers[:-40]:
    layer.trainable = False

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower Learning rate
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 111s 108ms/step - accuracy: 0.5910 - loss: 1.4024 - val_accuracy: 0.5187 - val_loss: 1.8303 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 140s 105ms/step - accuracy: 0.7359 - loss: 1.0039 - val_accuracy: 0.5248 - val_loss: 1.7229 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 89s 98ms/step - accuracy: 0.7958 - loss: 0.8581 - val_accuracy: 0.5305 - val_loss: 1.6150 - learning_rate: 1.0000e-05
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8364 - loss: 0.7618 - val_accuracy: 0.5362 - val_loss: 1.6074 - learning_rate: 1.0000e-05
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 98ms/step - accuracy: 0.8664 - loss: 0.6914 - val_accuracy: 0.5685 - val_loss: 1.4994 - learning_rate: 1.0000e-05
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.8833 - loss: 0.6555 - val_accuracy: 0.5876 - val_loss: 1.3972 - learning_rate: 1.0000e-05
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 149s 105

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

473/473 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.7664 - loss: 1.0991
Driver-wise Test Accuracy: 0.7663963437080383


In [ ]:
base_model.trainable = True

# Freeze most layers, train last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,720,736 (25.64 MB)

 Trainable params: 2,046,986 (7.81 MB)

 Non-trainable params: 579,776 (2.21 MB)

 Optimizer params: 4,093,974 (15.62 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # lower LR
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 103s 102ms/step - accuracy: 0.9297 - loss: 0.5494 - val_accuracy: 0.6325 - val_loss: 1.3118 - learning_rate: 1.0000e-05
Epoch 2/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 95s 105ms/step - accuracy: 0.9395 - loss: 0.5276 - val_accuracy: 0.6165 - val_loss: 1.3772 - learning_rate: 1.0000e-05
Epoch 3/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 105ms/step - accuracy: 0.9432 - loss: 0.5113 - val_accuracy: 0.6144 - val_loss: 1.3664 - learning_rate: 3.0000e-06
Epoch 4/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 87s 97ms/step - accuracy: 0.9445 - loss: 0.5108 - val_accuracy: 0.6196 - val_loss: 1.3536 - learning_rate: 3.0000e-06
Epoch 5/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 94s 104ms/step - accuracy: 0.9470 - loss: 0.5043 - val_accuracy: 0.6223 - val_loss: 1.3447 - learning_rate: 9.0000e-07
Epoch 6/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - accuracy: 0.9495 - loss: 0.4976 - val_accuracy: 0.6205 - val_loss: 1.3415 - learning_rate: 9.0000e-07
Epoch 7/20
840/840 ━━━━━━━━━━━━━━━━━━━━ 143s 1

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise Test Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 25s 70ms/step - accuracy: 0.6428 - loss: 1.2920
Driver-wise Test Accuracy: 0.6428319215774536


In [ ]:
model.save("/content/data/models/mobileNet_baseline_final.keras")